In [4]:
import pandas as pd

file_path = r"C:\Users\DELL\OneDrive\Desktop\Project sem 2\data\Fatality rate\fatality rate_variables_GTD.xlsx"

# Read both sheets
nkill_df = pd.read_excel(file_path, sheet_name="nkill")
nwound_df = pd.read_excel(file_path, sheet_name="nwound")



In [5]:
# Convert nkill
nkill_long = nkill_df.melt(
    id_vars=["Country"], 
    var_name="Year", 
    value_name="nkill"
)

# Convert nwound
nwound_long = nwound_df.melt(
    id_vars=["Country"], 
    var_name="Year", 
    value_name="nwound"
)

In [6]:
panel_df = pd.merge(
    nkill_long,
    nwound_long,
    on=["Country", "Year"],
    how="inner"
)

In [7]:
panel_df["Year"] = panel_df["Year"].astype(int)
panel_df = panel_df.sort_values(by=["Country", "Year"])

In [9]:
# Avoid division by zero
panel_df["fatality_rate"] = panel_df["nkill"] / (panel_df["nkill"] + panel_df["nwound"])

# Handle cases where nkill + nwound = 0
panel_df["fatality_rate"] = panel_df["fatality_rate"].fillna(0)

In [12]:

panel_df

,Country,Year,nkill,nwound,fatality_rate
0,Afghanistan,1970,NaN,NaN,0.000000
204,Afghanistan,1971,NaN,NaN,0.000000
408,Afghanistan,1972,NaN,NaN,0.000000
612,Afghanistan,1973,0.0,1.0,0.000000
816,Afghanistan,1974,NaN,NaN,0.000000
...,...,...,...,...,...
9383,Zimbabwe,2016,NaN,NaN,0.000000
9587,Zimbabwe,2017,0.0,1.0,0.000000
9791,Zimbabwe,2018,2.0,47.0,0.040816
9995,Zimbabwe,2019,0.0,0.0,0.000000


In [23]:
import pandas as pd
from functools import reduce

file_path = r"C:\Users\DELL\OneDrive\Desktop\Project sem 2\data\Fatality rate\WGI_fatality_1.xlsx"

xls = pd.ExcelFile(file_path)
dfs = []

for sheet in xls.sheet_names:
    df = pd.read_excel(file_path, sheet_name=sheet)
    
    # Rename 'value' to indicator name (sheet name)
    df.rename(columns={"value": sheet.lower()}, inplace=True)
    
    # Keep required columns
    df = df[["Country", "Year", sheet.lower()]]
    
    # Clean data
    df["Country"] = df["Country"].str.strip().str.lower()
    df["Year"] = df["Year"].astype(int)
    
    # Ensure uniqueness (VERY IMPORTANT)
    df = df.drop_duplicates(subset=["Country", "Year"])
    
    dfs.append(df)

# Merge all sheets
wgi_merged = reduce(
    lambda left, right: pd.merge(left, right, on=["Country", "Year"], how="inner"),
    dfs
)

# Sort
wgi_merged = wgi_merged.sort_values(by=["Country", "Year"]).reset_index(drop=True)

# Check
print(wgi_merged.shape)
print(wgi_merged.head())

(4973, 8)
       Country  Year  voice and accountability  political stability  \
0  afghanistan  1996                    0.0625               0.0625   
1  afghanistan  1998                    0.0625               0.0625   
2  afghanistan  2000                    0.0625               0.0625   
3  afghanistan  2002                    0.1875               0.0625   
4  afghanistan  2003                    0.2500               0.1250   

   government effectiveness  regulatory quality  rule of law  \
0                     0.000            0.000000       0.1875   
1                     0.000            0.000000       0.1875   
2                     0.000            0.000000       0.1875   
3                     0.125            0.166667       0.1875   
4                     0.125            0.250000       0.1875   

   control of corruption  
0                  0.000  
1                  0.000  
2                  0.000  
3                  0.125  
4                  0.125  


In [24]:
# Ensure same format before merging

# Clean panel_df (terrorism data)
panel_df["Country"] = panel_df["Country"].astype(str).str.strip().str.lower()
panel_df["Year"] = panel_df["Year"].astype(int)

# Clean WGI data
wgi_merged["Country"] = wgi_merged["Country"].astype(str).str.strip().str.lower()
wgi_merged["Year"] = wgi_merged["Year"].astype(int)

# Merge
final_df = pd.merge(
    panel_df,
    wgi_merged,
    on=["Country", "Year"],
    how="left"   # 🔥 recommended
)

In [25]:
final_df

,Country,Year,nkill,nwound,fatality_rate,voice and accountability,political stability,government effectiveness,regulatory quality,rule of law,control of corruption
0,afghanistan,1970,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
1,afghanistan,1971,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
2,afghanistan,1972,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
3,afghanistan,1973,0.0,1.0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
4,afghanistan,1974,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
10195,zimbabwe,2016,NaN,NaN,0.000000,0.185000,0.3125,0.0,0.107143,0.125,0.0
10196,zimbabwe,2017,0.0,1.0,0.000000,0.188667,0.3125,0.0,0.107143,0.100,0.0
10197,zimbabwe,2018,2.0,47.0,0.040816,0.188667,0.1875,0.0,0.107143,0.150,0.0
10198,zimbabwe,2019,0.0,0.0,0.000000,0.105333,0.1250,0.0,0.107143,0.125,0.0
